# Análisis y pronóstico de la PEA de Lima Metropolitana

**Periodo analizado:** enero de 2020 a diciembre de 2025  
**Frecuencia:** mensual  
**Unidad:** miles de personas

Este notebook estudia la evolución de la **Población Económicamente Activa (PEA) de Lima Metropolitana**, identifica sus principales cambios y compara distintos modelos de series de tiempo.

## Contenido

1. Entorno y carga de datos  
2. Análisis descriptivo de la serie  
3. Componentes y propiedades temporales  
4. División en entrenamiento y prueba  
5. Modelos de suavización  
6. Modelos Box-Jenkins: ARIMA y SARIMA  
7. Comparación, selección y pronóstico  
8. Conclusiones y recomendaciones

## 1. Entorno y carga de datos

La base contiene dos variables:

- `periodo`: fecha mensual.
- `PEA`: PEA de Lima Metropolitana, expresada en miles de personas.

Para descargar la data se puede acceder a la fuente del BCRP que es:

`https://estadisticas.bcrp.gob.pe/estadisticas/series/mensuales/resultados/PN38050GM/html/2020-1/2025-12/`

In [1]:
# Librerías principales
import warnings

from statsmodels.tools.sm_exceptions import (ConvergenceWarning, InterpolationWarning)

# Ocultar advertencias específicas de Statsmodels
warnings.simplefilter("ignore", ConvergenceWarning)
warnings.simplefilter("ignore", InterpolationWarning)

import itertools
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

from scipy.stats import jarque_bera
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf
from statsmodels.tsa.holtwinters import (SimpleExpSmoothing, Holt, ExponentialSmoothing)
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [2]:
# Cargamos la base de datos
serie = (pd.read_excel('../../2data_local_me/data_lima_labor_force_forecasting/data_pea.xlsx', parse_dates=["periodo"]).set_index("periodo")["PEA"].asfreq("MS"))

serie.head()

periodo
2020-01-01   5,378.30
2020-02-01   5,377.70
2020-03-01   5,230.30
2020-04-01   4,015.60
2020-05-01   2,934.30
Freq: MS, Name: PEA, dtype: float64

In [3]:
# Control básico de calidad
control = pd.DataFrame({
    "Indicador": ["Observaciones", "Fecha inicial", "Fecha final", "Valores perdidos", "Fechas duplicadas"],
    "Resultado": [
        len(serie), serie.index.min().date(), 
        serie.index.max().date(),
        int(serie.isna().sum()), 
        int(serie.index.duplicated().sum())
    ]
})
control

,Indicador,Resultado
0,Observaciones,72
1,Fecha inicial,2020-01-01
2,Fecha final,2025-12-01
3,Valores perdidos,0
4,Fechas duplicadas,0


La base tiene **72 observaciones mensuales completas**, desde enero de 2020 hasta diciembre de 2025. No se observan meses faltantes ni fechas duplicadas.

## 2. Análisis descriptivo de la serie temporal

### 2.1 Evolución mensual

In [4]:
datos_grafico = serie.rename("PEA").reset_index()

fig = px.line(
    datos_grafico, x="periodo", y="PEA", markers=True,
    title="PEA de Lima Metropolitana, enero de 2020-diciembre de 2025",
    labels={"periodo": "Periodo", "PEA": "Miles de personas"})

fig.update_layout(hovermode="x unified")
fig.show()

### Interpretación

La serie presenta tres etapas claramente diferenciadas:

1. **Caída extraordinaria en 2020.** La PEA pasó de aproximadamente 5378,3 miles de personas en enero a 2625,3 miles en junio. Este quiebre coincide con las restricciones y la paralización de actividades ocasionadas por la pandemia de COVID-19.
2. **Recuperación entre el segundo semestre de 2020 y 2022.** La PEA aumentó con rapidez después del mínimo de junio de 2020, aunque la recuperación no fue completamente uniforme.
3. **Crecimiento más estable entre 2023 y 2025.** La serie mantuvo una trayectoria ascendente y alcanzó 6189,4 miles de personas en diciembre de 2025.


### 2.2 Comportamiento por año

In [5]:
resumen_anual = serie.resample("YE").agg(["mean", "min", "max", "first", "last"])
resumen_anual.index = resumen_anual.index.year
resumen_anual.columns = ["Promedio", "Mínimo", "Máximo", "Enero", "Diciembre"]
resumen_anual["Variación dic./dic. (%)"] = resumen_anual["Diciembre"].pct_change() * 100
resumen_anual

,Promedio,Mínimo,Máximo,Enero,Diciembre,Variación dic./dic. (%)
periodo,,,,,,
2020,"4,367.66","2,625.30","5,378.30","5,378.30","5,055.55",NaN
2021,"5,096.83","4,841.12","5,326.50","5,102.52","5,326.50",5.36
2022,"5,427.31","5,320.39","5,510.40","5,320.39","5,473.92",2.77
2023,"5,581.41","5,506.16","5,682.55","5,537.31","5,682.55",3.81
2024,"5,820.52","5,705.13","5,898.56","5,705.13","5,879.23",3.46
2025,"5,949.02","5,819.35","6,189.37","5,819.35","6,189.37",5.28


In [6]:
fig = px.bar(
    resumen_anual.reset_index(), x="periodo", y="Promedio",
    title="Promedio anual de la PEA de Lima Metropolitana",
    labels={"periodo": "Año", "Promedio": "Miles de personas"},
    text_auto=".0f")

fig.show()

El promedio de 2020 es considerablemente menor debido al choque sanitario. Desde 2021, los promedios anuales aumentan de manera continua. Además, el crecimiento entre diciembre de 2024 y diciembre de 2025 fue cercano al **5,3 %**.

## 3. Componentes y propiedades temporales

### 3.1 Descomposición de la serie

La descomposición aditiva separa la serie en tres componentes:

$$
Y_t = T_t + S_t + R_t
$$

Donde:

- $Y_t$: valor observado de la PEA.
- $T_t$: componente de tendencia.
- $S_t$: componente estacional.
- $R_t$: residuo o movimiento no explicado.

Se utiliza la descomposición **STL**, para visualizar la influencia de observaciones sobre la tendencia, la estacionalidad y los residuos.

In [7]:
# Aplicar la descomposición STL
# period=12 porque la serie es mensual
stl = STL(serie, period=12, robust=True).fit()

# Muestra una tabla únicamente con los tres componentes
componentes = pd.DataFrame({"Tendencia": stl.trend, "Estacionalidad": stl.seasonal, "Residuo": stl.resid}).reset_index()
componentes.head()

,periodo,Tendencia,Estacionalidad,Residuo
0,2020-01-01,"4,143.75",38.66,"1,195.90"
1,2020-02-01,"4,200.89",107.13,"1,069.68"
2,2020-03-01,"4,257.90",177.25,795.15
3,2020-04-01,"4,314.73",-287.60,-11.53
4,2020-05-01,"4,371.36",-7.22,"-1,429.84"


#### 3.1.1 Tendencia de la serie

In [8]:
fig_tendencia = px.line(
    componentes,
    x="periodo",
    y="Tendencia",
    title="Componente de tendencia de la PEA de Lima Metropolitana",
    labels={"periodo": "Periodo", "Tendencia": "Miles de personas"})
fig_tendencia.update_layout(hovermode="x unified", template="plotly_white", height=400, showlegend=False)
fig_tendencia.show()

Este gráfico muestra la evolución subyacente de largo plazo de la PEA, suavizando las variaciones mensuales y los movimientos extraordinarios. En la descomposición STL robusta, la tendencia presenta una trayectoria progresivamente ascendente, mientras que la caída abrupta observada durante 2020 es recogida principalmente por el componente residual.

#### 3.1.2 Estacionalidad de la serie

In [9]:
fig_estacionalidad = px.line(
    componentes,
    x="periodo",
    y="Estacionalidad",
    title="Componente estacional de la PEA de Lima Metropolitana",
    labels={"periodo": "Periodo", "Estacionalidad": "Variación estacional"})
fig_estacionalidad.add_hline(y=0, line_dash="dash")
fig_estacionalidad.update_layout(hovermode="x unified", template="plotly_white", height=400, showlegend=False)
fig_estacionalidad.show()

Los valores positivos indican meses en los que la PEA suele ubicarse por encima de su tendencia. Los valores negativos representan meses en los que suele encontrarse por debajo.

#### 3.1.3 Residuos de la serie

In [10]:
fig_residuo = px.line(
    componentes,
    x="periodo",
    y="Residuo",
    title="Componente residual de la PEA de Lima Metropolitana",
    labels={"periodo": "Periodo","Residuo": "Variación no explicada"})
fig_residuo.add_hline(y=0, line_dash="dash")
fig_residuo.update_layout(hovermode="x unified", template="plotly_white", height=400, showlegend=False)
fig_residuo.show()

El residuo recoge los movimientos que no son explicados por la tendencia ni por la estacionalidad. Los valores muy alejados de cero pueden señalar periodos atípicos, como los asociados al choque de la COVID-19 durante 2020.

### Interpretación de la descomposición

- La **tendencia** muestra una trayectoria subyacente progresivamente ascendente a lo largo del periodo analizado. Debido al uso de una descomposición STL robusta, la caída extraordinaria de 2020 no se incorpora completamente en la tendencia.
- La **estacionalidad** presenta variaciones relativamente pequeñas frente a los movimientos extraordinarios observados en 2020 y frente a la trayectoria de largo plazo.
- Los **residuos** de mayor magnitud se concentran principalmente alrededor del periodo de la pandemia, lo que refleja el carácter excepcional (atípico) de estos movimientos dentro de la serie.

### 3.2 Estacionariedad: pruebas ADF y KPSS

Una serie de tiempo es estacionaria cuando sus principales propiedades estadísticas, como la media, la varianza y la estructura de autocorrelación, se mantienen aproximadamente constantes a lo largo del tiempo.

Para evaluar esta propiedad se utilizan dos pruebas estadisticas:

- **Prueba Dickey-Fuller aumentada (ADF):** su hipótesis nula, $H_0$, establece que la serie presenta una raíz unitaria y, por tanto, no es estacionaria.
- **Prueba Kwiatkowski-Phillips-Schmidt-Shin (KPSS):** su hipótesis nula, $H_0$, establece que la serie es estacionaria alrededor de una constante.

Las reglas de decisión, considerando un nivel de significancia de $\alpha=0.05$, son:

- En la prueba ADF, un p-valor menor que $0.05$ permite rechazar la hipótesis de raíz unitaria.
- En la prueba KPSS, un p-valor mayor o igual que $0.05$ indica que no se rechaza la hipótesis de estacionariedad.

Las pruebas se aplican primero a la serie original, denominada serie en niveles. Posteriormente, se evalúa la serie en primera diferencia:

$$
\Delta Y_t = Y_t-Y_{t-1}
$$

Donde:

- $Y_t$: valor de la PEA en el periodo $t$.
- $Y_{t-1}$: valor de la PEA en el periodo anterior.
- $\Delta Y_t$: cambio de la PEA entre dos meses consecutivos.

El análisis conjunto de las pruebas ADF y KPSS permite obtener una conclusión más consistente, debido a que ambas utilizan hipótesis nulas opuestas.

In [11]:
def pruebas_estacionariedad(x, nombre):
    x = x.dropna()
    adf_resultado = adfuller(x, regression="c", autolag="AIC")
    kpss_resultado = kpss(x, regression="c", nlags="auto")
    adf_pvalor = adf_resultado[1]
    kpss_pvalor = kpss_resultado[1]
    if adf_pvalor < 0.05 and kpss_pvalor >= 0.05:
        conclusion = "Estacionaria"  
    elif adf_pvalor >= 0.05 and kpss_pvalor < 0.05:
        conclusion = "No estacionaria"   
    else:
        conclusion = "Resultado no concluyente"
    return {
        "Serie evaluada": nombre,
        "Estadístico ADF": adf_resultado[0],
        "p-valor ADF": adf_pvalor,
        "Estadístico KPSS": kpss_resultado[0],
        "p-valor KPSS": kpss_pvalor,
        "Conclusión": conclusion}

pruebas = pd.DataFrame([pruebas_estacionariedad(serie, "Serie original"),
                        pruebas_estacionariedad(serie.diff(),"Serie en primera diferencia")])

columnas_numericas = ["Estadístico ADF", "p-valor ADF","Estadístico KPSS","p-valor KPSS"]
pruebas[columnas_numericas] = pruebas[columnas_numericas].round(4)
pruebas

,Serie evaluada,Estadístico ADF,p-valor ADF,Estadístico KPSS,p-valor KPSS,Conclusión
0,Serie original,0.06,0.96,0.96,0.01,No estacionaria
1,Serie en primera diferencia,-3.71,0.00,0.05,0.10,Estacionaria


### Interpretación

En serie original, la prueba ADF no rechaza la existencia de raíz unitaria y la prueba KPSS rechaza la estacionariedad. En cambio, después de aplicar una primera diferencia, ADF rechaza la raíz unitaria y KPSS no rechaza la estacionariedad.

Por tanto, existe evidencia para trabajar con un orden de diferenciación no estacional igual a:

$$
d=1
$$

### 3.3 Funciones ACF y PACF

La función de autocorrelación muestra la relación de la serie con sus rezagos. La autocorrelación parcial mide la relación directa con cada rezago, controlando los rezagos intermedios.

Estas funciones sirven como orientación para los órdenes autorregresivos y de medias móviles de los modelos ARIMA y SARIMA.

In [12]:
serie_diferenciada = serie.diff().dropna()
rezagos = 24
valores_acf = acf(serie_diferenciada, nlags=rezagos, fft=False)
valores_pacf = pacf(serie_diferenciada, nlags=rezagos, method="ywm")
limite = 1.96 / np.sqrt(len(serie_diferenciada))

In [13]:
fig = go.Figure()
fig.add_trace(go.Bar(x=list(range(rezagos + 1)), y=valores_acf, name="ACF"))
fig.add_hline(y=limite, line_dash="dash")
fig.add_hline(y=-limite, line_dash="dash")
fig.update_layout(title="ACF de la primera diferencia", xaxis_title="Rezago", yaxis_title="Autocorrelación")
fig.show()

In [14]:
fig = go.Figure()
fig.add_trace(go.Bar(x=list(range(rezagos + 1)), y=valores_pacf, name="PACF"))
fig.add_hline(y=limite, line_dash="dash")
fig.add_hline(y=-limite, line_dash="dash")
fig.update_layout(title="PACF de la primera diferencia", xaxis_title="Rezago", yaxis_title="Autocorrelación parcial")
fig.show()

La ACF y la PACF muestran principalmente dependencia de corto plazo, con algunos rezagos que superan los límites de referencia. No se observa un patrón anual claramente dominante en estos gráficos, por lo que se utilizan principalmente como orientación. Debido a que la muestra de entrenamiento contiene solo 60 observaciones, los órdenes de los modelos ARIMA y SARIMA se determinan mediante una búsqueda pequeña y controlada, evitando estructuras excesivamente complejas.

## 4. División en entrenamiento y prueba

La división temporal será:

- **Entrenamiento:** enero de 2020 a diciembre de 2024.
- **Prueba:** enero de 2025 a diciembre de 2025.

Esta división permite estimar los modelos con información conocida hasta 2024 y comprobar si sus pronósticos para 2025 son cercanos a los valores realmente observados.

In [15]:
entrenamiento = serie.loc[:"2024-12-01"]
prueba = serie.loc["2025-01-01":]

print(f"Observaciones de entrenamiento: {len(entrenamiento)}")
print(f"Observaciones de prueba: {len(prueba)}")

Observaciones de entrenamiento: 60
Observaciones de prueba: 12


In [16]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=entrenamiento.index, y=entrenamiento, mode="lines+markers", name="Entrenamiento"))
fig.add_trace(go.Scatter(x=prueba.index, y=prueba, mode="lines+markers", name="Prueba"))
fig.add_vline(x=pd.Timestamp("2025-01-01"), line_dash="dash")
fig.update_layout(
    title="División temporal de la serie",
    xaxis_title="Periodo", yaxis_title="Miles de personas",
    hovermode="x unified")

fig.show()

### Métricas de evaluación

$$
MAE=\frac{1}{n}\sum_{t=1}^{n}\left|Y_t-\widehat{Y}_t\right|
$$

$$
MAPE=\frac{100}{n}\sum_{t=1}^{n}\left|\frac{Y_t-\widehat{Y}_t}{Y_t}\right|
$$

$$
RMSE=\sqrt{\frac{1}{n}\sum_{t=1}^{n}\left(Y_t-\widehat{Y}_t\right)^2}
$$

- **MAE:** error absoluto promedio en miles de personas.
- **MAPE:** error promedio en términos porcentuales.
- **RMSE:** penaliza con mayor fuerza los errores grandes.

La selección final se realizará principalmente con las métricas del periodo de prueba.

In [17]:
def metricas(reales, estimados):
    datos = pd.concat([reales.rename("real"), estimados.rename("estimado")], axis=1).dropna()
    error = datos["real"] - datos["estimado"]
    return {
        "MAE": np.mean(np.abs(error)),
        "MAPE (%)": np.mean(np.abs(error / datos["real"])) * 100,
        "RMSE": np.sqrt(np.mean(error ** 2))
    }

pronosticos_test = {}
ajustes_train = {}
modelos_ajustados = {}

## 5. Modelos de suavización

### 5.1 Promedio móvil simple

El promedio móvil simple de orden \(k\) asigna el mismo peso a las últimas \(k\) observaciones:

$$
\widehat{Y}_{t+1}=\frac{1}{k}\sum_{i=0}^{k-1}Y_{t-i}
$$

Se evaluarán ventanas de 3 y 6 meses. Para producir varios meses hacia adelante, el procedimiento se aplica de forma recursiva.

In [18]:
def pronostico_pms(serie_train, horizonte, ventana, indice):
    historial = list(serie_train.astype(float))
    predicciones = []
    for _ in range(horizonte):
        valor = np.mean(historial[-ventana:])
        predicciones.append(valor)
        historial.append(valor)
    return pd.Series(predicciones, index=indice)

for ventana in [3, 6]:
    nombre = f"PMS ({ventana} meses)"
    ajustes_train[nombre] = entrenamiento.shift(1).rolling(ventana).mean()
    pronosticos_test[nombre] = pronostico_pms(entrenamiento, len(prueba), ventana, prueba.index)

### 5.2 Promedio móvil ponderado

El promedio móvil ponderado otorga mayor importancia a las observaciones recientes:

$$
\widehat{Y}_{t+1}=\sum_{i=1}^{k}w_iY_{t-k+i},
\qquad \sum_{i=1}^{k}w_i=1
$$

Se utilizarán tres meses con pesos \(0.20\), \(0.30\) y \(0.50\), desde el dato más antiguo hasta el más reciente.

In [19]:
def pronostico_pmp(serie_train, horizonte, pesos, indice):
    pesos = np.asarray(pesos, dtype=float)
    pesos = pesos / pesos.sum()
    historial = list(serie_train.astype(float))
    predicciones = []
    for _ in range(horizonte):
        valor = np.dot(historial[-len(pesos):], pesos)
        predicciones.append(valor)
        historial.append(valor)
    return pd.Series(predicciones, index=indice)

pesos = np.array([0.20, 0.30, 0.50])
ajustes_train["PMP (3 meses)"] = entrenamiento.shift(1).rolling(3).apply(lambda x: np.dot(x, pesos), raw=True)
pronosticos_test["PMP (3 meses)"] = pronostico_pmp(entrenamiento, len(prueba), pesos, prueba.index)

### 5.3 Suavización exponencial simple

La suavización exponencial simple actualiza el nivel de la serie mediante:

$$
L_t=\alpha Y_t+(1-\alpha)L_{t-1},
\qquad 0<\alpha<1
$$

Este modelo es más apropiado cuando la serie fluctúa alrededor de un nivel sin tendencia ni estacionalidad pronunciadas. Se incluye para comprobar su desempeño, aunque la PEA presenta una trayectoria creciente después de 2020.

In [20]:
modelo_ses = SimpleExpSmoothing(entrenamiento, initialization_method="estimated").fit(optimized=True)
modelos_ajustados["SES"] = modelo_ses
ajustes_train["SES"] = modelo_ses.fittedvalues
pronosticos_test["SES"] = modelo_ses.forecast(len(prueba))

### 5.4 Método de Holt

Holt incorpora un nivel y una tendencia:

$$
L_t=\alpha Y_t+(1-\alpha)(L_{t-1}+B_{t-1})
$$

$$
B_t=\beta(L_t-L_{t-1})+(1-\beta)B_{t-1}
$$

$$
\widehat{Y}_{t+h}=L_t+hB_t
$$


In [21]:
modelo_holt = Holt(entrenamiento, initialization_method="estimated").fit(optimized=True)
modelos_ajustados["Holt"] = modelo_holt
ajustes_train["Holt"] = modelo_holt.fittedvalues
pronosticos_test["Holt"] = modelo_holt.forecast(len(prueba))

### 5.5 Holt-Winters

El modelo aditivo combina nivel, tendencia y estacionalidad:

$$
\widehat{Y}_{t+h}=L_t+hB_t+S_{t+h-12}
$$
para horizontes de hasta 12 meses.

Se utiliza la forma aditiva porque la amplitud de las oscilaciones estacionales no aumenta de manera proporcional al nivel de la PEA.

In [22]:
modelo_hw = ExponentialSmoothing(
    entrenamiento,
    trend="add",
    seasonal="add",
    seasonal_periods=12,
    initialization_method="estimated"
).fit(optimized=True, use_brute=True)

modelos_ajustados["Holt-Winters aditivo"] = modelo_hw
ajustes_train["Holt-Winters aditivo"] = modelo_hw.fittedvalues
pronosticos_test["Holt-Winters aditivo"] = modelo_hw.forecast(len(prueba))

## 6. Modelos Box-Jenkins: ARIMA y SARIMA

### 6.1 Modelo ARIMA

Un modelo ARIMA se representa de manera general como:

$$
\phi(B)(1-B)^dY_t = \theta(B)\varepsilon_t
$$

Donde:

- $p$: número de términos autorregresivos.
- $d$: número de diferencias necesarias para alcanzar la estacionariedad.
- $q$: número de términos de medias móviles.

Debido a los resultados de las pruebas de estacionariedad, se fija $d=1$. Posteriormente, se prueban combinaciones de $p$ y $q$ entre 0 y 3, y se selecciona el modelo que presenta el menor valor del criterio de información de Akaike, denominado $\mathrm{AIC}$.

In [23]:
resultados_arima = []

for p, q in itertools.product(range(4), range(4)):
    try:
        ajuste = SARIMAX(
            entrenamiento,
            order=(p, 1, q),
            trend="n",
            enforce_stationarity=False,
            enforce_invertibility=False
        ).fit(disp=False, maxiter=300)

        # Solo se incorporan modelos cuya estimación haya convergido
        if ajuste.mle_retvals.get("converged", False):
            resultados_arima.append({"p": p,"d": 1,"q": q,"AIC": ajuste.aic,"modelo": ajuste})

    except Exception:
        pass

tabla_arima = (pd.DataFrame(resultados_arima).sort_values("AIC").reset_index(drop=True))
tabla_arima.drop(columns="modelo").head(5)

,p,d,q,AIC
0,3,1,3,660.42
1,2,1,3,676.14
2,0,1,3,679.12
3,1,1,3,680.29
4,2,1,2,685.17


In [24]:
mejor_arima = tabla_arima.iloc[0]
ajuste_arima = mejor_arima["modelo"]
nombre_arima = f"ARIMA({int(mejor_arima.p)},1,{int(mejor_arima.q)})"

modelos_ajustados[nombre_arima] = ajuste_arima
ajustes_train[nombre_arima] = ajuste_arima.get_prediction(
    start=entrenamiento.index[0], end=entrenamiento.index[-1], dynamic=False
).predicted_mean
pronosticos_test[nombre_arima] = ajuste_arima.get_forecast(len(prueba)).predicted_mean

print(f"Modelo ARIMA seleccionado por AIC: {nombre_arima}")

Modelo ARIMA seleccionado por AIC: ARIMA(3,1,3)


### 6.2 Modelo SARIMA

El modelo SARIMA amplía el modelo ARIMA al incorporar una estructura estacional. Se representa mediante la notación:

$$
SARIMA(p,d,q)(P,D,Q)_s
$$

Su expresión general es:

$$
\Phi(B^s)\phi(B)(1-B)^d(1-B^s)^D Y_t
=
\Theta(B^s)\theta(B)\varepsilon_t
$$

Donde:

- $p$: número de términos autorregresivos no estacionales.
- $d$: número de diferencias no estacionales.
- $q$: número de términos de medias móviles no estacionales.
- $P$: número de términos autorregresivos estacionales.
- $D$: número de diferencias estacionales.
- $Q$: número de términos de medias móviles estacionales.
- $s$: periodicidad de la serie.
- $B$: operador de rezago.
- $\varepsilon_t$: término de error aleatorio.

Debido a que la serie presenta una frecuencia mensual, se establece una periodicidad estacional de:

$$
s=12
$$

Para identificar el modelo, se evalúan combinaciones reducidas de:

$$
p,q \in \{0,1,2\}
$$

y:

$$
P,Q \in \{0,1\}
$$

Asimismo, se fija $d=1$ y $D=1$. La búsqueda se limita a modelos de baja complejidad porque el periodo de entrenamiento contiene solamente 60 observaciones mensuales, correspondientes a enero de 2020-diciembre de 2024.

Finalmente, entre los modelos estimados, se selecciona aquel que presenta el menor valor del criterio de información de Akaike, $\mathrm{AIC}$.

In [25]:
resultados_sarima = []

for p, q, P, Q in itertools.product(range(3), range(3), range(2), range(2)):
    if p == q == P == Q == 0:
        continue

    try:
        ajuste = SARIMAX(
            entrenamiento,
            order=(p, 1, q),
            seasonal_order=(P, 1, Q, 12),
            trend="n",
            enforce_stationarity=False,
            enforce_invertibility=False
        ).fit(disp=False, maxiter=300)

        # Solo se incorporan modelos cuya estimación haya convergido
        if ajuste.mle_retvals.get("converged", False):
            resultados_sarima.append({"p": p,"d": 1,"q": q,"P": P,"D": 1,"Q": Q,"s": 12,"AIC": ajuste.aic, "modelo": ajuste})

    except Exception:
        pass

tabla_sarima = (pd.DataFrame(resultados_sarima).sort_values("AIC").reset_index(drop=True))
tabla_sarima.drop(columns="modelo").head(5)

,p,d,q,P,D,Q,s,AIC
0,2,1,2,1,1,0,12,349.18
1,2,1,1,1,1,0,12,360.63
2,2,1,0,1,1,0,12,362.32
3,1,1,2,1,1,0,12,367.59
4,1,1,0,1,1,0,12,374.07


In [26]:
mejor_sarima = tabla_sarima.iloc[0]
ajuste_sarima = mejor_sarima["modelo"]
nombre_sarima = (
    f"SARIMA({int(mejor_sarima.p)},1,{int(mejor_sarima.q)})"
    f"({int(mejor_sarima.P)},1,{int(mejor_sarima.Q)})[12]"
)

modelos_ajustados[nombre_sarima] = ajuste_sarima
ajustes_train[nombre_sarima] = ajuste_sarima.get_prediction(
    start=entrenamiento.index[0], end=entrenamiento.index[-1], dynamic=False
).predicted_mean
pronosticos_test[nombre_sarima] = ajuste_sarima.get_forecast(len(prueba)).predicted_mean

print(f"Modelo SARIMA seleccionado por AIC: {nombre_sarima}")

Modelo SARIMA seleccionado por AIC: SARIMA(2,1,2)(1,1,0)[12]


## 7. Comparación, selección y pronóstico

### 7.1 Métricas dentro de muestra y periodo de prueba

Para comparar el ajuste dentro de muestra bajo un mismo periodo, se define un inicio común posterior al mayor periodo de inicialización o burn-in requerido por los modelos Box-Jenkins seleccionados. Asimismo, se considera como mínimo un ciclo anual de 12 meses debido a la presencia de modelos con estructura estacional mensual. De este modo, se evita comparar predicciones iniciales que todavía pueden estar condicionadas por la inicialización del modelo.

Las métricas dentro de muestra permiten evaluar qué tan bien se ajusta cada modelo a los datos utilizados durante su estimación. Sin embargo, no deben utilizarse como el único criterio de selección, ya que un buen ajuste sobre los datos de entrenamiento no garantiza una adecuada capacidad predictiva.

Las métricas correspondientes al periodo de prueba, comprendido entre enero y diciembre de 2025, son las más importantes para evaluar el desempeño predictivo, porque comparan los pronósticos con observaciones que no fueron utilizadas durante la estimación de los modelos.

Se utilizan las siguientes métricas:

- **MAE:** error absoluto promedio, expresado en miles de personas.
- **MAPE:** error porcentual absoluto promedio.
- **RMSE:** raíz del error cuadrático medio, que penaliza con mayor intensidad los errores grandes.

La selección final se realiza según el menor RMSE en el periodo de prueba. El MAE y el MAPE se reportan como métricas complementarias para evaluar la magnitud de los errores.

In [27]:
filas_metricas = []

burn_comun = max(
    12,
    int(ajuste_arima.loglikelihood_burn),
    int(ajuste_sarima.loglikelihood_burn))

inicio_comun_train = entrenamiento.index[burn_comun]

for nombre in pronosticos_test:
    met_train = metricas(
        entrenamiento.loc[inicio_comun_train:],
        ajustes_train[nombre].reindex(entrenamiento.index).loc[inicio_comun_train:]
    )
    met_test = metricas(prueba, pronosticos_test[nombre])

    filas_metricas.append({
        "Modelo": nombre,
        "MAE entrenamiento": met_train["MAE"],
        "MAPE entrenamiento (%)": met_train["MAPE (%)"],
        "RMSE entrenamiento": met_train["RMSE"],
        "MAE test": met_test["MAE"],
        "MAPE test (%)": met_test["MAPE (%)"],
        "RMSE test": met_test["RMSE"]
    })

comparacion = pd.DataFrame(filas_metricas).sort_values("RMSE test").reset_index(drop=True)
comparacion

,Modelo,MAE entrenamiento,MAPE entrenamiento (%),RMSE entrenamiento,MAE test,MAPE test (%),RMSE test
0,"SARIMA(2,1,2)(1,1,0)[12]",32.21,0.57,40.04,95.23,1.60,106.38
1,PMP (3 meses),40.59,0.72,49.47,109.70,1.81,146.43
2,PMS (3 meses),46.10,0.82,56.20,109.57,1.81,146.93
3,SES,32.91,0.58,38.34,109.84,1.81,147.02
4,"ARIMA(3,1,3)",35.65,0.63,42.49,112.75,1.86,156.42
5,PMS (6 meses),65.33,1.16,79.27,109.32,1.80,157.96
6,Holt-Winters aditivo,98.65,1.75,127.88,220.89,3.71,247.84
7,Holt,41.72,0.74,49.99,204.56,3.37,275.04


In [28]:
fig = px.bar(
    comparacion.sort_values("RMSE test", ascending=False),
    x="RMSE test", y="Modelo", orientation="h",
    title="Comparación de modelos según RMSE en el periodo de prueba",
    labels={"RMSE test": "RMSE en 2025", "Modelo": "Modelo"},
    text_auto=".1f")

fig.show()

In [29]:
# Comparación visual de los pronósticos para 2025
fig = go.Figure()
fig.add_trace(go.Scatter(x=prueba.index, y=prueba, mode="lines+markers", name="PEA real 2025"))

for nombre in comparacion.head(5)["Modelo"]:
    fig.add_trace(go.Scatter(
        x=prueba.index,
        y=pronosticos_test[nombre],
        mode="lines",
        name=nombre
    ))

fig.update_layout(
    title="Valores reales y pronósticos de los cinco modelos con menor RMSE",
    xaxis_title="Periodo", yaxis_title="Miles de personas",
    hovermode="x unified")

fig.show()

In [30]:
mejor_modelo = comparacion.iloc[0]
nombre_mejor_modelo = mejor_modelo["Modelo"]

print(f"Mejor modelo en el test: {nombre_mejor_modelo}")
print(f"MAE test: {mejor_modelo['MAE test']:.2f} miles de personas")
print(f"MAPE test: {mejor_modelo['MAPE test (%)']:.2f}%")
print(f"RMSE test: {mejor_modelo['RMSE test']:.2f} miles de personas")

# Las secciones de diagnóstico y pronóstico final están implementadas
# específicamente para un modelo SARIMA.
if nombre_mejor_modelo != nombre_sarima:
    raise ValueError(
        f"El mejor modelo fue {nombre_mejor_modelo}, pero las secciones 7.2 y 7.3 "
        "están implementadas específicamente para SARIMA. "
        "Debe revisarse la selección del modelo final."
    )

modelo_final_nombre = nombre_sarima
ajuste_modelo_final = ajuste_sarima

Mejor modelo en el test: SARIMA(2,1,2)(1,1,0)[12]
MAE test: 95.23 miles de personas
MAPE test: 1.60%
RMSE test: 106.38 miles de personas


### Interpretación de la comparación

El modelo con menor RMSE en 2025 es el que reproduce mejor el comportamiento real del periodo de prueba. En esta serie, el modelo SARIMA obtiene el mejor desempeño general y supera a todos los modelos evaluados.

Un MAPE cercano a 1,6 % significa que, en promedio, la distancia porcentual entre la PEA observada y la estimada durante 2025 fue reducida. Sin embargo, la comparación corresponde a un único año de prueba, por lo que el resultado debe interpretarse como evidencia favorable y no como garantía de precisión permanente.

Dado que SARIMA presenta el menor RMSE en el periodo de prueba, se adopta como modelo final para realizar el diagnóstico de residuos y elaborar el pronóstico de 2026.

### 7.2 Diagnóstico de residuos del modelo SARIMA seleccionado

Los residuos deberían aproximarse a ruido blanco:

$$
\varepsilon_t=Y_t-\widehat{Y}_t
$$

Se revisan:

- gráfico temporal de residuos;
- ACF residual;
- prueba de Ljung-Box para autocorrelación;
- prueba de Jarque-Bera para normalidad.

Para realizar el diagnóstico, se excluye el periodo inicial de burn-in determinado por el propio modelo. De este modo, se evita que los residuos correspondientes a la etapa de inicialización influyan en la evaluación del comportamiento residual.

In [31]:
burn_final = int(ajuste_modelo_final.loglikelihood_burn)
residuos_mejor = ajuste_modelo_final.resid.iloc[burn_final:].dropna()

fig = px.line(
    residuos_mejor.rename("Residuo").reset_index(),
    x="periodo", y="Residuo",
    title=f"Residuos de {modelo_final_nombre}",
    labels={"periodo": "Periodo", "Residuo": "Error"})
fig.add_hline(y=0, line_dash="dash")
fig.show()

In [32]:
acf_residuos = acf(residuos_mejor, nlags=min(18, len(residuos_mejor) // 2), fft=False)
limite_residuos = 1.96 / np.sqrt(len(residuos_mejor))
fig = go.Figure()
fig.add_trace(go.Bar(x=list(range(len(acf_residuos))), y=acf_residuos))
fig.add_hline(y=limite_residuos, line_dash="dash")
fig.add_hline(y=-limite_residuos, line_dash="dash")
fig.update_layout(title="ACF de los residuos", xaxis_title="Rezago", yaxis_title="Autocorrelación")
fig.show()

In [33]:
ljung_box = acorr_ljungbox(residuos_mejor, lags=[6, 12], return_df=True)
jb = jarque_bera(residuos_mejor)

diagnostico = pd.DataFrame({
    "Prueba": ["Ljung-Box (6 rezagos)", "Ljung-Box (12 rezagos)", "Jarque-Bera"],
    "Estadístico": [ljung_box.loc[6, "lb_stat"], ljung_box.loc[12, "lb_stat"], jb.statistic],
    "p-valor": [ljung_box.loc[6, "lb_pvalue"], ljung_box.loc[12, "lb_pvalue"], jb.pvalue]
})
diagnostico

,Prueba,Estadístico,p-valor
0,Ljung-Box (6 rezagos),2.96,0.81
1,Ljung-Box (12 rezagos),9.70,0.64
2,Jarque-Bera,2.08,0.35


### Interpretación del diagnóstico

Una vez excluido el periodo inicial de burn-in, la prueba de Ljung-Box no encuentra evidencia significativa de autocorrelación residual en los rezagos evaluados. Esto indica que el modelo recoge adecuadamente la dependencia temporal relevante.

Asimismo, la prueba de Jarque-Bera no permite rechazar la hipótesis de normalidad de los residuos al nivel de significancia del 5 %. En conjunto, los resultados del diagnóstico son favorables para el modelo SARIMA seleccionado.

### 7.3 Pronóstico para 2026

Dado que el modelo SARIMA presentó el menor RMSE en el periodo de prueba de 2025, se adopta como modelo final y se vuelve a estimar utilizando las 72 observaciones disponibles entre enero de 2020 y diciembre de 2025. A partir de esta reestimación se obtiene el pronóstico para los 12 meses de 2026, junto con un intervalo de confianza del 95 %.

In [34]:
# Se conserva la estructura del modelo SARIMA seleccionado y se reestima con toda la serie
orden_final = (int(mejor_sarima.p), 1, int(mejor_sarima.q))
orden_estacional_final = (int(mejor_sarima.P), 1, int(mejor_sarima.Q), 12)

modelo_final = SARIMAX(
    serie,
    order=orden_final,
    seasonal_order=orden_estacional_final,
    trend="n",
    enforce_stationarity=False,
    enforce_invertibility=False
).fit(disp=False, maxiter=300)

resultado_2026 = modelo_final.get_forecast(steps=12)
pronostico_2026 = resultado_2026.predicted_mean.rename("Pronóstico")
intervalos_2026 = resultado_2026.conf_int(alpha=0.05)
intervalos_2026.columns = ["Límite inferior 95%", "Límite superior 95%"]

tabla_pronostico_2026 = pd.concat([pronostico_2026, intervalos_2026], axis=1)
tabla_pronostico_2026

,Pronóstico,Límite inferior 95%,Límite superior 95%
2026-01-01,"6,117.30","6,034.46","6,200.13"
2026-02-01,"6,133.32","6,014.81","6,251.83"
2026-03-01,"6,152.49","5,995.72","6,309.27"
2026-04-01,"6,186.92","5,994.66","6,379.19"
2026-05-01,"6,150.00","5,932.09","6,367.91"
2026-06-01,"6,159.37","5,923.23","6,395.52"
2026-07-01,"6,231.24","5,978.12","6,484.36"
2026-08-01,"6,297.22","6,025.04","6,569.39"
2026-09-01,"6,338.64","6,046.93","6,630.36"
2026-10-01,"6,393.99","6,085.30","6,702.68"


In [35]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=serie.index, y=serie, mode="lines", name="PEA observada"))
fig.add_trace(go.Scatter(x=pronostico_2026.index, y=pronostico_2026, mode="lines+markers", name="Pronóstico 2026"))
fig.add_trace(go.Scatter(
    x=list(intervalos_2026.index) + list(intervalos_2026.index[::-1]),
    y=list(intervalos_2026["Límite superior 95%"]) + list(intervalos_2026["Límite inferior 95%"][::-1]),
    fill="toself", line=dict(width=0), name="Intervalo de confianza 95%"))

fig.update_layout(
    title=f"Pronóstico de la PEA para 2026 con {modelo_final_nombre}",
    xaxis_title="Periodo", yaxis_title="Miles de personas",
    hovermode="x unified")

fig.show()

### Interpretación del pronóstico

El pronóstico para 2026 muestra una trayectoria generalmente ascendente de la PEA, aunque con algunas variaciones mensuales. El modelo estima aproximadamente 6117,3 miles de personas en enero y 6501,6 miles en diciembre de 2026. Asimismo, los intervalos de confianza se amplían progresivamente conforme aumenta el horizonte de pronóstico, reflejando una mayor incertidumbre en las estimaciones de los meses más alejados.

## 8. Conclusiones y recomendaciones

### Conclusiones

1. La PEA de Lima Metropolitana registró un quiebre extraordinario durante 2020. El valor mínimo se presentó en junio, con aproximadamente **2625,3 miles de personas**.
2. Después del choque del COVID-19, la serie mostró una recuperación progresiva. En diciembre de 2025 alcanzó aproximadamente **6189,4 miles de personas**, nivel superior al observado al inicio de 2020.
3. La serie no es estacionaria en niveles, pero su primera diferencia presenta evidencia favorable de estacionariedad.
4. Con entrenamiento entre enero de 2020 y diciembre de 2024, el modelo **SARIMA** presentó el mejor desempeño para pronosticar los valores observados entre enero y diciembre de 2025.
5. El error porcentual absoluto medio (MAPE) del mejor modelo comparado fue cercano a **1,6 %** en el periodo de prueba.

### Recomendaciones

- Mantener las observaciones correspondientes a 2020, ya que reflejan un choque económico real y no un error en la base de datos.
- Actualizar el modelo a medida que se incorporen nuevos meses de información y volver a evaluar su precisión predictiva.
- Interpretar los pronósticos conjuntamente con sus intervalos de confianza, especialmente debido a la presencia del choque excepcional observado en 2020.
- Si en el futuro se dispone de una serie histórica más extensa previa a 2020, evaluar modelos con variables de intervención que permitan estimar de manera más formal el efecto de la pandemia.